# Rooted in Wealth: Street Tree Maturity and Neighbourhood Income in Vancouver

Kaiya Sjoholm  
August 10, 2026

## Introduction

Vancouver is widely celebrated for its green spaces, but how evenly its street trees are distributed and whether that distribution favours wealthier neighbourhoods is less well understood. Understanding this pattern matters because it could inform city planting investment; genus diversity also matters since a canopy concentrated in only a couple of genera carries real ecological risk. 

Using a subset of the City of Vancouver's open street trees dataset, containing 5,000 randomly sampled trees across 22 neighbourhoods alongside Statistics Canada's 2016 Census, this analysis asks to what extent the maturity of Vancouver’s street trees tracks neighbourhood wealth, and whether recent planting has begun to close that gap.

This analysis focuses on the trees that line boulevards and sidewalks, rather than Vancouver's parks or forested areas, and asks not only how mature those trees are, but how resilient the mix of species planted across the city really is.

Throughout the report, tree maturity is measured using trunk diameter, since direct age data is not available for most trees in this dataset. Trunk diameter is not a perfect stand-in for age, since different genera reach different maximum sizes; the extent to which species composition alone explains the diameter gap is discussed further below.

Neighbourhood wealth is measured using average household income by local area from the 2016 census, using the most recent income data at this geographic level.

The street trees dataset was obtained from the City of Vancouver's Open Data Portal and follows an Open Government License. The census dataset, also from the same portal, must be credited to Statistics Canada, Census 2016 rather than under the same licence. These datasets required cleaning before use in this analysis, including fixing inconsistent neighbourhood name formatting, reshaping the census data into a usable structure, and correcting data types.


## Analysis

## About the Datasets

This analysis uses 9 columns from the trees and census datasets. `neighbourhood_name` is used to group the trees, income, and planting data together across all four figures. `mean_diameter` is used as a proxy for tree maturity, since age data is not available. `avg_household_income` represents wealth, using 2016 census data as a measurable indicator.

`genus_name` and `unique_genera` are used to assess diversity and resilience. `pct_single_detached` is used as a proxy for what type of dwellings populate the neighbourhoods, impacting the space for trees to grow. `dominant_genus` identifies the most common tree genus in each neighbourhood, used to assess whether certain neighbourhoods have a less diverse planting of trees. 

`dominant_genus_pct` shows the percentage that genus represents, which matters because a genus at 20% is a much weaker "dominant" story than one at 45%, so the raw category name alone can be misleading. `date_planted` allows analysis of planting trends over time. 

Of these, `date_planted` is the most significant gap, with 2,637 out of 5,000 recorded entries missing a recorded planting date. These missing dates are not random; they concentrate in wealthier neighbourhoods. `diameter` also has a maximum of 71 inches compared to a mean of 12 and a 75th percentile of 18. Additionally, some trees are recorded with a diameter of 0, which is likely a recording error and were filtered out in this analysis. 

Several of these, such as `mean_diameter` and `dominant_genus_pct`, are calculated in the preparation steps below.

### Loading the data

Two datasets are loaded: a random sample of 5,000 City of Vancouver street trees, and the 2016 census local area profiles. The census file uses latin-1 encoding and has four rows of notes above its column headers, so both settings are specified when reading it.

In [1]:
import pandas as pd
import altair as alt


In [2]:
census_df = pd.read_csv('../data/CensusLocalAreaProfiles2016.csv', encoding='latin-1', header=4)
trees_df = pd.read_csv('../data/small_unique_vancouver.csv', index_col=0)

### Housing type by neighbourhood

The census file stores each measure as a row and each neighbourhood as a column. To find the exact label for single-detached homes, all rows mentioning "detached" are searched first. The single-detached and total dwellings rows are then reshaped with `melt()` so there is one row per neighbourhood, merged, converted to integers, and used to calculate the percentage of single-detached homes. The city-wide totals, Vancouver CSD and Vancouver CMA, are removed since they aren't neighbourhoods.

In [3]:
detached_rows = census_df[census_df['Variable'].str.contains('detached', case=False, na=False)]

In [4]:
single_detached_row = census_df[census_df['Variable'].str.strip() == 'Single-detached house']

In [5]:
single_detached_long = single_detached_row.melt(
    id_vars = ['Variable', 'ID'],
    var_name = 'neighbourhood',
    value_name = 'single_detached_homes'
)

In [6]:
total_dwellings_row = census_df[census_df['Variable'].str.strip() == 'Total - Occupied private dwellings by structural type of dwelling - 100% data']
total_dwellings_long = total_dwellings_row.melt(
    id_vars = ['Variable', 'ID'],
    var_name = 'neighbourhood',
    value_name = 'total_dwellings'
)

In [7]:
dwellings_combined = single_detached_long.merge(total_dwellings_long, on='neighbourhood')

In [8]:
dwellings_combined['single_detached_homes'] = dwellings_combined['single_detached_homes'].astype(int)
dwellings_combined['total_dwellings'] = dwellings_combined['total_dwellings'].astype(int)

In [9]:
dwellings_combined = dwellings_combined.assign(
    pct_single_detached = dwellings_combined['single_detached_homes'] / dwellings_combined['total_dwellings'] * 100
)

In [10]:
dwellings_combined = dwellings_combined[~dwellings_combined['neighbourhood'].isin(['Vancouver CSD', 'Vancouver CMA'])]

In [11]:
dwellings_combined = dwellings_combined[['neighbourhood', 'pct_single_detached']]

### Tree measures by neighbourhood

Trees with a recorded diameter of 0 are removed first, since these are likely recording errors. For each neighbourhood, the mean diameter, number of trees, number of unique genera, and dominant genus are then calculated.

In [12]:
trees_df_valid_diameter = trees_df[trees_df['diameter'] > 0]

In [13]:
diameter_by_neigh = trees_df_valid_diameter.groupby('neighbourhood_name')['diameter'].mean().reset_index()

In [14]:
diameter_by_neigh = diameter_by_neigh.rename(columns={'diameter': 'mean_diameter'})

In [15]:
n_trees = trees_df.groupby('neighbourhood_name').size().reset_index(name='n_trees')

In [16]:
unique_genera = trees_df.groupby('neighbourhood_name')['genus_name'].nunique().reset_index(name='unique_genera')

In [17]:
genus_counts = trees_df.groupby(['neighbourhood_name', 'genus_name']).size().reset_index(name='genus_count')


In [18]:
genus_counts_sorted = genus_counts.sort_values(['neighbourhood_name', 'genus_count'], ascending=[True, False])

dominant_genus = genus_counts_sorted.groupby('neighbourhood_name').head(1)


In [19]:
dominant_genus = dominant_genus.rename(columns={'genus_name': 'dominant_genus', 'genus_count': 'dominant_genus_count'})

In [20]:
dominant_genus = dominant_genus.merge(n_trees, on='neighbourhood_name')

In [21]:
dominant_genus = dominant_genus.assign(
    dominant_genus_pct = dominant_genus['dominant_genus_count'] / dominant_genus['n_trees'] * 100
)

### Household income by neighbourhood

Average household income is taken from the census using the same reshaping steps as housing type. Neighbourhood names in both census tables are renamed and stripped of extra spaces so they match the tree data exactly. Without this step, the merge would silently drop any neighbourhood whose name differs by a single space.

In [22]:
avg_income_row = census_df[census_df['Variable'].str.strip() == 'Average total income of households in 2015 ($)']

In [23]:
income_long = avg_income_row.melt(
    id_vars = ['Variable', 'ID'],
    var_name = 'neighbourhood',
    value_name = 'avg_household_income'
)

In [24]:
dwellings_combined = dwellings_combined.rename(columns={'neighbourhood': 'neighbourhood_name'})
income_long = income_long.rename(columns={'neighbourhood': 'neighbourhood_name'})

In [25]:
income_long = income_long[['neighbourhood_name', 'avg_household_income']]

In [26]:
dwellings_combined['neighbourhood_name'] = dwellings_combined['neighbourhood_name'].str.strip()

In [27]:
income_long['neighbourhood_name'] = income_long['neighbourhood_name'].str.strip()

### Combining and checking

All neighbourhood-level measures are merged into one table, `neighbourhood_summary`. The checks that follow confirm it has one row for each of the 22 neighbourhoods and no missing values, and convert income to an integer for charting.

In [28]:
neighbourhood_summary = diameter_by_neigh.merge(unique_genera, on='neighbourhood_name')
neighbourhood_summary = neighbourhood_summary.merge(dominant_genus, on='neighbourhood_name')
neighbourhood_summary = neighbourhood_summary.merge(dwellings_combined, on='neighbourhood_name')
neighbourhood_summary = neighbourhood_summary.merge(income_long, on='neighbourhood_name')

In [29]:
neighbourhood_summary.shape

(22, 9)

In [30]:
neighbourhood_summary.isnull().sum()

neighbourhood_name      0
mean_diameter           0
unique_genera           0
dominant_genus          0
dominant_genus_count    0
n_trees                 0
dominant_genus_pct      0
pct_single_detached     0
avg_household_income    0
dtype: int64

In [31]:
neighbourhood_summary['avg_household_income'] = neighbourhood_summary['avg_household_income'].astype(int)

In [32]:
neighbourhood_summary['avg_household_income'].dtype

dtype('int64')

***Table 1: Summary statistics for the 22 neighbourhoods***

In [33]:
neighbourhood_summary.describe()

,mean_diameter,unique_genera,dominant_genus_count,n_trees,dominant_genus_pct,pct_single_detached,avg_household_income
count,22.000000,22.000000,22.000000,22.000000,22.000000,22.000000,22.000000
mean,12.349665,31.181818,61.818182,227.272727,27.653511,23.491450,107733.954545
std,1.891540,5.737399,20.129666,83.555047,5.110589,19.334926,47106.826927
min,7.480117,19.000000,13.000000,75.000000,17.333333,0.042802,46368.000000
25%,11.361837,27.000000,51.500000,175.250000,24.392045,6.653138,85503.000000
50%,12.522952,32.000000,66.000000,213.000000,27.966102,19.453452,90353.000000
75%,13.365382,34.750000,75.000000,286.250000,30.306330,36.402784,121129.000000
max,16.078115,46.000000,94.000000,384.000000,38.059701,64.565588,262760.000000


A few things stand out in Table 1. Income is skewed by a few wealthy neighbourhoods: the mean is about \$107,700, while the median is about \$90,400, and the highest income, \$262,760, is more than five times the lowest. Mean tree diameter ranges from 7.5 to 16.1 inches, but half of the neighbourhoods fall between 11.4 and 13.4 inches, so the extremes stand out. The share of single-detached homes varies even more widely, from under 1% to about 65%. Finally, neighbourhoods have between 75 and 384 sampled trees, so averages for neighbourhoods with fewer trees are less reliable.

## Figure 1: Mean Street Tree Diameter by Neighbourhood

This figure examines whether tree diameter varies across neighbourhoods and whether that variation tracks income.

In [34]:
def style_chart(chart, width=400, height=300):
    return (
        chart
        .properties(width=width, height=height)
        .configure_title(fontSize=18, font='sans-serif', anchor='start')
        .configure_axis(labelFontSize=12, titleFontSize=14, titleFontWeight='normal', labelColor='#333', titleColor='#1a1a1a', gridColor='#e8e8e8', gridOpacity=0.6)
        .configure_legend(titleFontSize=13, labelFontSize=11, titleFontWeight='normal')
    )


In [35]:
fig1 = alt.Chart(neighbourhood_summary, title='Figure 1: Mean Street Tree Diameter by Neighbourhood').mark_bar().encode(
    x=alt.X('mean_diameter:Q', title='Mean Tree Diameter (inches)'),
    y=alt.Y('neighbourhood_name:N', title='Neighbourhood', sort='-x'),
    color=alt.Color('avg_household_income:Q', title='Average household income (2015, $)', scale=alt.Scale(scheme='blues'), legend=alt.Legend(format='$,.0f', titleLimit=250)),
    tooltip=[
        alt.Tooltip('neighbourhood_name:N', title='Neighbourhood'),
        alt.Tooltip('mean_diameter:Q', title='Mean diameter (in)', format='.1f'),
        alt.Tooltip('avg_household_income:Q', title='Avg. household income', format='$,.0f')
    ]
)
fig1_styled = style_chart(fig1, height=500)

fig1_styled
                    

alt.Chart(...)

**Key Observations**

Figure 1 uses a sorted bar chart so neighbourhoods can be compared by rank, with colour showing average household income to see whether the two line up.

Mean diameter ranges from 16.1 inches in Dunbar-Southlands to 7.5 inches in Downtown, more than a twofold difference. Dunbar-Southlands, Kitsilano, and Shaughnessy have the largest trees, but the order doesn't follow income exactly: Shaughnessy has the highest income in the dataset yet ranks third, while Kitsilano, with a mid-range income, ranks second. Overall, neighbourhoods in the top half by income average 12.9 inches in diameter, compared with 11.8 inches in the bottom half.

This suggests income and tree size are related, but not perfectly aligned. Figure 2 examines that relationship more directly.

## Figure 2: Income vs. Mean Street Tree Diameter

Figure 2 explores whether neighbourhood wealth predicts tree size on its own, or whether it depends on how much room the trees have to grow.

In [36]:
fig2 = alt.Chart(neighbourhood_summary, title='Figure 2: Income vs. Mean Street Tree Diameter').mark_circle(opacity=0.6).encode(
    x=alt.X('avg_household_income:Q', title='Average household income (2015, $)', scale=alt.Scale(zero=False, domain=[30000, 310000])),
    y=alt.Y('mean_diameter:Q', title='Mean diameter (inches)', scale=alt.Scale(zero=False)),
    size=alt.Size('pct_single_detached:Q', title='% single-detached homes', legend=alt.Legend(titleLimit=200)),
    color=alt.Color('pct_single_detached:Q', title='% single-detached homes', scale=alt.Scale(scheme='oranges'), legend=None),
    tooltip=[   
        alt.Tooltip('neighbourhood_name:N', title='Neighbourhood'),
        alt.Tooltip('mean_diameter:Q', title='Mean diameter (in)', format='.1f'),
        alt.Tooltip('avg_household_income:Q', title='Avg. household income', format='$,.0f'),
        alt.Tooltip('pct_single_detached:Q', title='% Single-detached', format='.1f')
    ]
)

outliers = neighbourhood_summary[neighbourhood_summary['neighbourhood_name'].isin(['Downtown', 'West End', 'Shaughnessy', 'Dunbar-Southlands', 'Kitsilano'])]

labels = alt.Chart(outliers).mark_text(align='left', dx=7, dy=-9, fontSize=10, fontWeight='bold').encode(
    x='avg_household_income:Q',
    y='mean_diameter:Q',
    text='neighbourhood_name:N'
)

fig2_combined = (fig2 + labels).resolve_legend(size='independent')

fig2_styled = style_chart(fig2_combined, width=900, height=400)

fig2_styled

                    

alt.LayerChart(...)

In [37]:
neighbourhood_summary['avg_household_income'].corr(neighbourhood_summary['mean_diameter'])

np.float64(0.44170026011582747)

**Key Observations**

Figure 2 uses a scatter plot to show the relationship between income and mean tree diameter directly, with point size and colour adding a third variable, the percentage of single-detached homes.

The correlation between income and mean diameter is 0.44, a moderate positive relationship. The two highest-income neighbourhoods, Shaughnessy and Dunbar-Southlands, also have the highest share of single-detached homes, at about 65% and 62%, and both have large trees. However, housing type doesn't explain the pattern on its own either. Downtown has a mid-range income, almost no single-detached homes, and the smallest trees in the city at 7.5 inches. Yet West End, with similar housing and a lower income, averages 12.8 inches, above the city average, and Kitsilano, where only 6% of homes are single-detached, has the second-largest trees at 15.1 inches.

This suggests that neither income nor housing type fully explains tree size. Exceptions like Kitsilano and West End point to other factors, such as neighbourhood age or species mix, which are discussed below.

## Figure 3: Planting Activity by Year and Neighbourhood

Figure 3 examines whether recent planting has gone to neighbourhoods that already have mature trees, or to neighbourhoods that have fewer mature trees.

In [38]:
planting = trees_df[trees_df['date_planted'].notna()].copy()
planting['year_planted'] = pd.to_datetime(planting['date_planted']).dt.year
planting_counts = planting.groupby(['neighbourhood_name', 'year_planted']).size().reset_index(name='n_planted')

In [39]:
planting_counts = planting_counts.merge(
    neighbourhood_summary[['neighbourhood_name', 'avg_household_income']],
    on='neighbourhood_name'
)

In [40]:
sorted_neighbourhoods = neighbourhood_summary.sort_values('avg_household_income', ascending=False)['neighbourhood_name'].tolist()

fig3 = alt.Chart(planting_counts, title='Figure 3: Planting Activity by Year and Neighbourhood').mark_rect().encode(
    x=alt.X('year_planted:O', title='Year planted'),
    y=alt.Y('neighbourhood_name:N', title='Neighbourhood', sort=sorted_neighbourhoods),
    color=alt.Color('n_planted:Q', title='Trees planted', scale=alt.Scale(scheme='blues')),
    tooltip=[
        alt.Tooltip('neighbourhood_name:N', title='Neighbourhood'),
        alt.Tooltip('year_planted:O', title='Year'),
        alt.Tooltip('n_planted:Q', title='Trees planted')
    ]
)
fig3_styled = style_chart(fig3, width=700, height=500)
fig3_styled



alt.Chart(...)

In [41]:
planted_share = planting.groupby('neighbourhood_name').size() / trees_df.groupby('neighbourhood_name').size() * 100
planted_share.sort_values(ascending=False)

neighbourhood_name
Hastings-Sunrise            59.705882
Oakridge                    58.421053
Renfrew-Collingwood         56.250000
Sunset                      54.045307
Arbutus-Ridge               53.714286
Victoria-Fraserview         53.082192
Riley Park                  52.439024
South Cambie                50.000000
Grandview-Woodland          49.740933
Kensington-Cedar Cottage    48.000000
Marpole                     47.881356
Killarney                   47.000000
West Point Grey             46.590909
Downtown                    44.444444
Strathcona                  42.666667
Kerrisdale                  42.460317
Mount Pleasant              40.201005
Dunbar-Southlands           36.741214
Shaughnessy                 36.283186
Kitsilano                   33.828996
West End                    33.070866
Fairview                    29.104478
dtype: float64

In [42]:
share_income = planted_share.reset_index(name='planted_share').merge(
    neighbourhood_summary[['neighbourhood_name', 'avg_household_income']],
    on='neighbourhood_name'
)
median_income = share_income['avg_household_income'].median()
print(share_income['planted_share'].corr(share_income['avg_household_income']))

share_income.groupby(share_income['avg_household_income'] >= median_income)['planted_share'].mean()

-0.2557016995575849


avg_household_income
False    48.331292
True     44.002537
Name: planted_share, dtype: float64

**Key Observations**

Figure 3 uses a heatmap because it shows two categories at once, year and neighbourhood, with colour showing how many trees were planted. Neighbourhoods are sorted by income, highest at the top, so any income pattern appears as a difference between the top and bottom of the chart.

Recorded planting peaked in 1996 at 133 trees and dropped sharply after 2014, to just 6 trees in 2016. Lower-income neighbourhoods, including Hastings-Sunrise, Renfrew-Collingwood, Victoria-Fraserview, and Kensington-Cedar Cottage, show the densest and most consistent planting, while higher-income neighbourhoods such as Shaughnessy, Dunbar-Southlands, West Point Grey, and Kerrisdale show less.

Part of this difference reflects sample size, since those four lower-income neighbourhoods are also among the most heavily sampled, with between 292 and 384 trees each. Adjusting for this by looking at the share of each neighbourhood's sampled trees with a recorded planting date, the difference becomes small: 48% on average in the lower-income half of neighbourhoods, compared with 44% in the higher-income half, and the correlation between income and this share is weak, at -0.26. There are also clear exceptions, such as Oakridge and Arbutus-Ridge, which have above-median incomes and high shares. As the Discussion notes, missing planting dates are concentrated in wealthier neighbourhoods, so this comparison cannot fully separate less planting from less record-keeping.

## Figure 4: Genus Diversity by Neighbourhood

Figure 4 examines how diverse each neighbourhood's street trees are, and how concentrated they are in a single genus.

In [43]:
sorted_by_genera = neighbourhood_summary.sort_values('unique_genera', ascending=False)['neighbourhood_name'].tolist()

In [44]:
neighbourhood_summary = neighbourhood_summary.assign(
    genus_pct_label=neighbourhood_summary['dominant_genus_pct'].round(1).astype(str) + '%'
)


fig4 = alt.Chart(neighbourhood_summary, title='Figure 4: Genus Diversity by Neighbourhood').mark_bar().encode(
    x=alt.X('unique_genera:Q', title='Unique genera'),
    y=alt.Y('neighbourhood_name:N', title='Neighbourhood', sort=sorted_by_genera),
    color=alt.Color('dominant_genus:N', title='Dominant genus'),
    tooltip=[
        alt.Tooltip('neighbourhood_name:N', title='Neighbourhood'),
        alt.Tooltip('unique_genera:Q', title='Unique genera'),
        alt.Tooltip('dominant_genus:N', title='Dominant genus'),
        alt.Tooltip('genus_pct_label:N', title='Dominant genus share')
    ]
)
fig4_styled = style_chart(fig4, width=700, height=500)

In [45]:
share_labels = alt.Chart(neighbourhood_summary).mark_text(align='left', dx=5, fontSize=10, color='black').encode(
    x=alt.X('unique_genera:Q'),
    y=alt.Y('neighbourhood_name:N', sort=sorted_by_genera),
    text=alt.Text('genus_pct_label:N')
)

fig4_combined = (fig4 + share_labels)
fig4_styled = style_chart(fig4_combined, width=750, height=500)
fig4_styled

alt.LayerChart(...)

In [46]:
neighbourhood_summary['unique_genera'].corr(neighbourhood_summary['n_trees'])

np.float64(0.8466049611051143)

**Key Observations**

Figure 4 uses a bar chart of unique genera per neighbourhood, coloured by dominant genus, with the dominant genus's share labelled on each bar. This shows both how many types of trees each neighbourhood has and how concentrated they are.

Across all 22 neighbourhoods, only two genera appear as the dominant genus: ACER (maple) in 14 neighbourhoods and PRUNUS (cherry and plum) in the other 8. However, the dominance is not always strong. The dominant genus makes up between 17% and 38% of trees, and in Killarney, PRUNUS at 19.5% only narrowly beats ACER at 18.5%, so the "dominant genus" label can overstate how concentrated a neighbourhood's trees actually are.

The number of unique genera should be read with caution. It has a correlation of 0.85 with the number of trees sampled in each neighbourhood, so neighbourhoods with more sampled trees show more genera largely because more trees were counted. Strathcona, for example, has the fewest genera at 19, but also the fewest sampled trees at 75. Comparing diversity fairly would require an equal number of trees from each neighbourhood.

## Discussion

Having examined all four figures, a consistent but imperfect pattern emerges: wealth tracks maturity, while recorded planting leans only slightly toward lower-income areas once sample size is accounted for.

**Limitations**

2,637 of 5,000 trees are missing a recorded planting date, and this gap is not random; it concentrates in the higher-income neighbourhoods. This means Figure 3 likely underrepresents actual planting activity in those neighbourhoods, which is one reason the planting finding is stated cautiously. However, diameter data has no missing values at all, and it confirms the same wealth-linked pattern, which suggests the overall finding is not simply a result of missing planting date records.

Sample size also varies between neighbourhoods, from 75 sampled trees in Strathcona to 384 in Renfrew-Collingwood. Because the sample was drawn at random, these counts largely reflect how many street trees each neighbourhood has, which depends on its physical size, street length, and housing layout. This affects any measure based on raw counts: larger neighbourhoods naturally show more recorded plantings and more unique genera. Figures 3 and 4 were checked against sample size, and their conclusions are worded with this in mind. Averages and percentages, as used in Figures 1 and 2, are not affected in the same way.

The income data reflects 2015 household income, collected as part of the 2016 census, while the tree planting dates in this dataset span 1989 to 2019. This means there is a roughly 26-year gap between when wealth was measured and when many of the trees analyzed were actually planted. As a result, we cannot assume 2015's income levels reflect income at the time earlier trees were planted.

Species composition was considered as an alternative explanation for the diameter gap; the genus-level comparison detailed below complicates rather than confirms this explanation. However, tree age and growing space remain difficult to separate: wealthier neighbourhoods tend to have more single-detached housing, so it is unclear whether larger diameters reflect older trees, more physical room to grow, or some combination of both. 

The original exploratory analysis suggested that PRUNUS's prevalence in high-diameter neighbourhoods like Dunbar-Southlands indicated a broader link between wealth, PRUNUS dominance, and larger trees. However, comparing all 22 neighbourhoods shows that ACER-dominant neighbourhoods average larger diameter (12.52 in vs. 12.05 in) and higher income (\$110,987 vs. \$102,042) than PRUNUS-dominant ones, the opposite of the original claim. Dunbar-Southlands is real, but it is a single outlier, not a general pattern.

**Concluding Remarks**

Across the four figures, this analysis found that neighbourhood wealth is linked to street tree maturity, but not as cleanly as expected. Shaughnessy illustrates this: despite having the highest income in the dataset, it ranked only third in mean tree diameter, showing that neighbourhood wealth does not fully determine tree size. Recorded planting leans only slightly toward lower-income neighbourhoods once sample size is accounted for, and missing planting dates limit how firmly even this can be concluded.

This did not fully match initial expectations: the original exploratory analysis suggested PRUNUS dominance was linked to wealth and larger trees, but the fuller comparison across all 22 neighbourhoods showed the opposite; ACER-dominant neighbourhoods had larger diameter and higher income. This was a reminder that a single striking example can create a misleading impression, and that it is always worth verifying across the full dataset. 

Future analysis could explore whether the age-vs-growing-space confound could be resolved with more detailed data. A fairer comparison of planting activity and genus diversity would also standardize by neighbourhood size, for example trees per square kilometre, per kilometre of street, or per dwelling. A small map showing neighbourhood locations for readers unfamiliar with Vancouver's geography could make these visualizations more accessible to a general audience.

## Dashboard

To explore these patterns further, the dashboard below combines Figure 1 and Figure 2 into a single interactive panel. Click on a bar in Figure 1 to highlight matching points in Figure 2. The dropdown filters both charts by ACER or PRUNUS, and the minimum and maximum diameter sliders narrow the view to neighbourhoods within a specific diameter range.

In [47]:
neighbourhood_select = alt.selection_point(fields=['neighbourhood_name'])

fig1_dash = fig1.add_params(neighbourhood_select).encode(
    opacity=alt.condition(neighbourhood_select, alt.value(1), alt.value(0.3))
)

fig2_dash = fig2_combined.encode(
    opacity=alt.condition(neighbourhood_select, alt.value(1), alt.value(0.3))
)
fig2_dash = fig2_dash.properties(width=650)

In [48]:
genus_options = ['ACER', 'PRUNUS']
genus_dropdown = alt.binding_select(options=genus_options + [None], labels=genus_options + ['All'], name='Dominant Genus: ')
genus_select = alt.selection_point(fields=['dominant_genus'], bind=genus_dropdown)

In [49]:
min_diameter = alt.param(value=7, bind=alt.binding_range(min=7, max=17, step=1, name='Min Diameter (inches): '))
max_diameter = alt.param(value=17, bind=alt.binding_range(min=7, max=17, step=1, name='Max Diameter (inches): '))

In [50]:
import warnings
warnings.filterwarnings('ignore', message='Automatically deduplicated')

fig1_final = fig1_dash.add_params(genus_select).transform_filter(genus_select)
fig2_final = fig2_dash.add_params(genus_select).transform_filter(genus_select)
fig1_final = fig1_final.add_params(min_diameter, max_diameter).transform_filter(
    (alt.datum.mean_diameter >= min_diameter) & (alt.datum.mean_diameter <= max_diameter)
)
fig2_final = fig2_final.add_params(min_diameter, max_diameter).transform_filter(
    (alt.datum.mean_diameter >= min_diameter) & (alt.datum.mean_diameter <= max_diameter)
)

dashboard_v2 = (fig1_final & fig2_final).resolve_scale(color='independent', size='independent')
dashboard_v2

alt.VConcatChart(...)

## References

- City of Vancouver Open Data Portal — [Street Trees Dataset](https://opendata.vancouver.ca/explore/dataset/public-trees/information/?disjunctive.common_name&disjunctive.species_name). Licensed under the Open Government License – Vancouver.
- City of Vancouver Open Data Portal — [Census Local Area Profiles 2016](https://opendata.vancouver.ca/explore/dataset/census-local-area-profiles-2016/information/). Data provided by Statistics Canada; must be credited to Statistics Canada, Census 2016.
- Course Modules - Data Visualization, UBC Key Capabilities in Data Science